# Stage 19A — HAR Identity and Source-Sequence Verification (No Google Drive)

**Purpose:** Recheck SHA-256 pinned public human hg38/chimp panTro6 BED+FASTA and export the **152 provisional** catalog ortholog pairs; summarize original GEO guide reference. Embedded inputs are the user-provided Stage16/16B ZIPs audited locally.

**What this does NOT do:** It does not independently confirm original Noble et al. exact HAR target intervals unless a primary-author original BED is added separately; it does not establish TF occupancy, lineage-specific substitutions, or HAR–gene causality. The supplementary Pal DpnII baits are *not* exact HAR intervals.

**Run:** Colab → Runtime → Run all. The script downloads only a few MB of public resources, uses NO Google Drive and creates one ZIP. For truly independent **original author** HAR interval mapping, obtain a primary author-verified four-column BED named `Noble_Original_HAR_hg38.bed` and place it in `/content/HAR_Stage19A/` before running the optional crosswalk cell; otherwise all original sequence identity labels remain unverified.


In [ ]:
# Stage 19A — No-Google-Drive pinned source verification
from pathlib import Path
from collections import Counter
import base64, csv, hashlib, io, json, time, urllib.request, zipfile

ROOT = Path('/content/HAR_Stage19A') if Path('/content').exists() else Path('HAR_Stage19A')
ROOT.mkdir(parents=True,exist_ok=True)
EMBEDDED_ZIP_B64 = """..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_ZIP_B64))) as z:
    for item in z.namelist():
        (ROOT/item).write_bytes(z.read(item))
print('Unpacked input files:', sorted(p.name for p in ROOT.iterdir()))

def table(name):
    with (ROOT/name).open(encoding='utf-8',newline='') as f:
        return list(csv.DictReader(f))

crosswalk=table('S1_All180_Stage19A_Crosswalk.csv')
coords=table('S2_152_Provisional_Ortholog_Coordinates.csv')
assert len(crosswalk)==180 and len(coords)==152
assert len({r['screen_HAR'] for r in crosswalk})==180
print('180-row crosswalk and 152 provisional pairs loaded; exact original author HAR coordinates not yet established')


In [ ]:
# Download only pinned public source files and enforce byte-for-byte SHA256 checks.
# This uses public HTTP resources, NOT your Google Drive.
checksums=json.loads((ROOT/'SOURCE_CHECKSUMS.json').read_text())
source_checks=[]
for item in checksums:
    if item['name'] == 'meme_file.txt':
        continue # Stage 19A maps sequences; PWM scoring is a later stage.
    fname=ROOT/item['name']; good=False; last=''
    if fname.is_file() and hashlib.sha256(fname.read_bytes()).hexdigest()==item['sha256']:
        good=True
    else:
        for attempt in range(3):
            try:
                req=urllib.request.Request(item['source'],headers={'User-Agent':'HAR-Stage19A/1.0 (research reproducibility)'})
                with urllib.request.urlopen(req,timeout=100) as r: data=r.read()
                digest=hashlib.sha256(data).hexdigest()
                if digest!=item['sha256']:
                    raise ValueError('SHA256 source drift: '+item['name']+': '+digest)
                if len(data)!=item['bytes']:
                    raise ValueError('Byte length changed: '+item['name'])
                fname.write_bytes(data);good=True;break
            except Exception as e:
                last=str(e);print('Attempt',attempt+1,'failed for',item['name'],repr(last[:160]));time.sleep(2)
    source_checks.append({'file':item['name'],'sha256_valid':good,'download_url':item['source'],'reported_sha256':item['sha256'],'error':last if not good else ''})
if not all(x['sha256_valid'] for x in source_checks):
    (ROOT/'S7_Source_Download_Status.json').write_text(json.dumps(source_checks,indent=2))
    raise RuntimeError('One or more pinned BED/FASTA sources failed checksum/download. DO NOT proceed with unverifiable sequences.')
print('Pinned public BED and FASTA sources verified:',len(source_checks))


In [ ]:
# Read exact 0-based-half-open BED coordinates and coordinate-keyed FASTA records.
def read_bed(path):
    rec={}
    for l in path.read_text().splitlines():
        if not l.strip() or l.startswith('#'): continue
        arr=l.split('\t')
        if len(arr)<4: continue
        chrom,start,end,name=arr[0],int(arr[1]),int(arr[2]),arr[3]
        if name in rec: raise ValueError('Duplicated public catalog BED identifier: '+name)
        rec[name]=(chrom,start,end)
    return rec

def read_fasta(path):
    result={}; key=None;buf=[]
    for raw in path.read_text().splitlines():
        if raw.startswith('>'):
            if key is not None:
                if key in result:raise ValueError('Duplicated FASTA header: '+key)
                result[key]=''.join(buf).upper()
            key=raw[1:].split()[0];buf=[]
        elif raw.strip():buf.append(raw.strip())
    if key is not None: result[key]=''.join(buf).upper()
    return result

bed_h=read_bed(ROOT/'hars_hg38.bed')
bed_c=read_bed(ROOT/'hars_panTro6.bed')
fa_h=read_fasta(ROOT/'human_hars.fasta')
fa_c=read_fasta(ROOT/'chimp_hars.fasta')
assert len(bed_h)>=3200 and len(bed_c)>=3200 and len(fa_h)>=3200 and len(fa_c)>=3200

manifest=[]
hlines=[];clines=[]
for r in coords:
    alias=r['screen_HAR']; vid=r['catalog_ID_not_original_name']
    h=(r['human_hg38_chrom'],int(r['human_start_0based']),int(r['human_end_exclusive']))
    c=(r['chimp_panTro6_chrom'],int(r['chimp_start_0based']),int(r['chimp_end_exclusive']))
    assert bed_h.get(vid)==h, 'Catalog hg38 coordinate mismatch: '+alias
    assert bed_c.get(vid)==c, 'Catalog panTro6 coordinate mismatch: '+alias
    hk=f'{h[0]}:{h[1]}-{h[2]}'
    ck=f'{c[0]}:{c[1]}-{c[2]}'
    hs,cs=fa_h.get(hk),fa_c.get(ck)
    assert hs is not None and cs is not None, f'Missing sequence: {alias} / {vid}'
    assert len(hs)==h[2]-h[1] and len(cs)==c[2]-c[1], 'FASTA length mismatch: '+alias
    assert set(hs)<=set('ACGTN') and set(cs)<=set('ACGTN'), 'Unexpected nucleotide code: '+alias
    hlines.extend([f'>{alias}|{vid}|hg38|{hk}|PROVISIONAL',hs])
    clines.extend([f'>{alias}|{vid}|panTro6|{ck}|PROVISIONAL',cs])
    manifest.append({'screen_HAR':alias,'catalog_id_provisional':vid,
       'human_coordinates':hk,'chimp_coordinates':ck,'human_length':len(hs),'chimp_length':len(cs),
       'human_seq_sha256':hashlib.sha256(hs.encode()).hexdigest(),
       'chimp_seq_sha256':hashlib.sha256(cs.encode()).hexdigest(),
       'original_author_exact_HAR_interval_confirmed':'NO',
       'biological_interpretation':'PROVISIONAL_CATALOG_SEQUENCE_NOT_ORIGINAL_TARGET_CERTIFIED'})
(ROOT/'S7_152_Human_CatalogSequences_PROVISIONAL.fa').write_text('\n'.join(hlines)+'\n')
(ROOT/'S8_152_Chimp_CatalogSequences_PROVISIONAL.fa').write_text('\n'.join(clines)+'\n')
with (ROOT/'S9_152_Sequence_Checksum_Manifest.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)
print('Exported 152 human and 152 chimp provisional sequences with matching pinned BED/FASTA hashes.')


In [ ]:
# OPTIONAL primary original author HAR BED crosswalk (hg38, BED4).
# To use this, place /content/HAR_Stage19A/Noble_Original_HAR_hg38.bed
# with columns: chromosome, start_0based, end_exclusive, exact_original_HAR_name.
# This must be a *primary author-verified* source, not a Pal bait fragment or secondary catalog.
primary=ROOT/'Noble_Original_HAR_hg38.bed'
def ov(a,b):
    if a[0]!=b[0]:return 0
    return max(0,min(a[2],b[2])-max(a[1],b[1]))
primary_results=[]
if primary.exists():
    original=read_bed(primary)
    for r in coords:
        alias=r['screen_HAR'];v=r['catalog_ID_not_original_name']
        cat=bed_h[v]; src=original.get(alias)
        if not src: status='ORIGINAL_HAR_NOT_IN_PRIMARY_BED'; f1=f2=None
        else:
            common=ov(cat,src);f1=round(common/(cat[2]-cat[1]),5);f2=round(common/(src[2]-src[1]),5)
            status='EXACT_COORDINATES_CONCORDANT' if cat==src else ('RECIPROCAL_95PCT_CONCORDANT' if min(f1,f2)>=0.95 else 'COORDINATE_DISCORDANT_OR_PARTIAL')
        primary_results.append({'screen_HAR':alias,'catalog_id':v,'primary_original_BED_found':bool(src),
          'fraction_catalog_covered':f1,'fraction_original_covered':f2,'coordinate_status':status,
          'independent_functional_validation':'NOT_PERFORMED'})
    with (ROOT/'S10_Original_ExactHAR_BED_Comparison.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(primary_results[0]));w.writeheader();w.writerows(primary_results)
    print(Counter(r['coordinate_status'] for r in primary_results))
else:
    print('Primary original author HAR BED not available: original element identity remains UNVERIFIED for all 180 HARs.')

# Download GEO feature reference for original guide names/sequences (NOT genomic HAR BED).
geo_url='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_feature_README.csv'
geo_status={'url':geo_url,'downloaded':False,'contains_original_exact_HAR_BED':False}
try:
    p=ROOT/'GSE270828_feature_README.csv'
    if not p.exists():
        req=urllib.request.Request(geo_url,headers={'User-Agent':'HAR-Stage19A/1.0'})
        with urllib.request.urlopen(req,timeout=100) as res:data=res.read()
        p.write_bytes(data)
    with p.open(newline='',encoding='utf-8-sig') as f:
        rr=csv.DictReader(f); cols=rr.fieldnames or []; example=[]
        n=0
        for rec in rr:
            n+=1
            if len(example)<2: example.append(rec)
    geo_status.update({'downloaded':True,'size_bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
                       'n_feature_rows':n,'columns':cols,'note':'sgRNA design/reference, NOT independently verified genomic HAR-element intervals'})
except Exception as e:
    geo_status['error']=str(e)
    print('GEO guide reference could not be retrieved; original element identity stays UNVERIFIED:',str(e)[:160])
(ROOT/'S11_GEO_Guide_Reference_Status.json').write_text(json.dumps(geo_status,indent=2))


In [ ]:
# Archive results for analysis in ChatGPT; download link appears below when Colab runs.
status={'stage':'19A','original_HARs':180,'provisional_sequence_pairs':152,
        'primary_original_HAR_BED_provided':primary.exists(),
        'original_HAR_identity_confirmed_without_primary_BED':False,
        'source_files_verified':source_checks,
        'GEO_feature_reference':geo_status,
        'note':'Human–chimp sequences are exact pinned public catalog sequences, not established originals from Noble design.'}
(ROOT/'STAGE19A_COLAB_STATUS.json').write_text(json.dumps(status,indent=2))
artifact=ROOT.parent/'HAR_Stage19A_Colab_Source_Verified_Results.zip'
with zipfile.ZipFile(artifact,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in ROOT.iterdir():
        if f.is_file() and f.name not in {'human_hars.fasta','chimp_hars.fasta','hars_hg38.bed','hars_panTro6.bed'}:
            z.write(f,f.name)
print('RESULT ZIP READY:',artifact,'size MB:',round(artifact.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(artifact))
except ImportError:
    print('Not in Google Colab; archive remains available at',artifact)
